# Install & Imports

In [1]:
from google.colab import drive
drive.mount('/users/')

Mounted at /users/


In [2]:
import os
import json
import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import tensorflow as tf
from tensorflow.keras import layers, models

# Dataset Paths

In [3]:
BASE = "/users/"

DATASETS = {
    "CIC_IIoT_2025":          os.path.join(BASE, "CIC_IIoT_2025"),
    "CIC_BCCC_NRC_IoMT_2024": os.path.join(BASE, "CIC_BCCC_NRC_IoMT_2024"),
    "CSE_CIC_IDS2018":        os.path.join(BASE, "CSE_CIC_IDS2018"),
    "FALCON_ID_Combined":     os.path.join(BASE, "Combined"),
}

MODELS_DIR = "/users/"
os.makedirs(MODELS_DIR, exist_ok=True)

# Reproducibility Setup

In [4]:
EPOCHS      = 10
BATCH_SIZE  = 2048
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# Data Loaders

In [5]:
def load_latent_splits(dataset_dir):
    X_train = np.load(os.path.join(dataset_dir, "train_latent.npy"))
    X_val   = np.load(os.path.join(dataset_dir, "val_latent.npy"))
    X_test  = np.load(os.path.join(dataset_dir, "test_latent.npy"))

    y_train = np.load(os.path.join(dataset_dir, "y_train.npy"))
    y_val   = np.load(os.path.join(dataset_dir, "y_val.npy"))
    y_test  = np.load(os.path.join(dataset_dir, "y_test.npy"))

    return X_train, X_val, X_test, y_train, y_val, y_test


def load_all_datasets():
    all_data = {}
    for name, path in DATASETS.items():
        print(f"\n📌 Loading: {name}")
        if name == "FALCON_ID_Combined":
            X_train = np.load(os.path.join(path, "combined_train_latent.npy"))
            X_val   = np.load(os.path.join(path, "combined_val_latent.npy"))
            X_test  = np.load(os.path.join(path, "combined_test_latent.npy"))

            y_train = np.load(os.path.join(path, "combined_y_train.npy"))
            y_val   = np.load(os.path.join(path, "combined_y_val.npy"))
            y_test  = np.load(os.path.join(path, "combined_y_test.npy"))
        else:
            X_train, X_val, X_test, y_train, y_val, y_test = load_latent_splits(path)

        print(f"✔ Shapes: {X_train.shape} {X_val.shape} {X_test.shape}")

        all_data[name] = {
            "X_train": X_train,
            "X_val":   X_val,
            "X_test":  X_test,
            "y_train": y_train,
            "y_val":   y_val,
            "y_test":  y_test,
        }
    return all_data


all_data = load_all_datasets()

# Quick overview
rows = []
for name, d in all_data.items():
    rows.append([
        name,
        d["X_train"].shape[0],
        d["X_val"].shape[0],
        d["X_test"].shape[0],
        d["X_train"].shape[1],
        len(np.unique(d["y_train"])),
    ])

summary_df = pd.DataFrame(
    rows, columns=["Dataset", "Train", "Val", "Test", "Latent_Dim", "Classes"]
)
print("\n🔎 Dataset Overview")
display(summary_df)


📌 Loading: CIC_IIoT_2025
✔ Shapes: (29424, 64) (6305, 64) (6306, 64)

📌 Loading: CIC_BCCC_NRC_IoMT_2024
✔ Shapes: (2369719, 64) (507797, 64) (507797, 64)

📌 Loading: CSE_CIC_IDS2018
✔ Shapes: (6737603, 64) (1443772, 64) (1443773, 64)

📌 Loading: FALCON_ID_Combined
✔ Shapes: (9136746, 64) (1957874, 64) (1957876, 64)

🔎 Dataset Overview


,Dataset,Train,Val,Test,Latent_Dim,Classes
0,CIC_IIoT_2025,29424,6305,6306,64,7
1,CIC_BCCC_NRC_IoMT_2024,2369719,507797,507797,64,15
2,CSE_CIC_IDS2018,6737603,1443772,1443773,64,15
3,FALCON_ID_Combined,9136746,1957874,1957876,64,37


# FALCON-ID Model

In [6]:
def build_falcon_id_model_v3(
    input_dim: int,
    num_classes: int,
    encoder_dim: int = 256,
    encoder_blocks: int = 3,
    lstm_units: int = 128,
    dropout_rate: float = 0.4
) -> tf.keras.Model:

    inputs = layers.Input(shape=(input_dim,), name="latent_input")

    # ---------------- Encoder (fully trainable) ----------------
    x = layers.LayerNormalization(name="enc_norm_in")(inputs)
    x = layers.Dense(encoder_dim, activation="gelu", name="enc_dense_0")(x)
    x = layers.BatchNormalization(name="enc_bn_0")(x)

    for i in range(encoder_blocks):
        skip = x
        y = layers.Dense(encoder_dim, activation="gelu", name=f"enc_block_{i+1}_dense_1")(x)
        y = layers.BatchNormalization(name=f"enc_block_{i+1}_bn_1")(y)
        y = layers.Dropout(dropout_rate, name=f"enc_block_{i+1}_drop_1")(y)
        y = layers.Dense(encoder_dim, name=f"enc_block_{i+1}_dense_2")(y)
        y = layers.BatchNormalization(name=f"enc_block_{i+1}_bn_2")(y)
        x = layers.Add(name=f"enc_block_{i+1}_add")([skip, y])
        x = layers.Activation("gelu", name=f"enc_block_{i+1}_act")(x)

    enc_out = layers.LayerNormalization(name="enc_norm_out")(x)  # (256,)

    # ---------------- Sequence tower (Conv1D + Bi-LSTM + Attn) ----------------
    # 256 --> (16, 16)
    seq = layers.Reshape((16, encoder_dim // 16), name="seq_reshape")(enc_out)

    # Conv1D feature extractor
    seq = layers.Conv1D(filters=64, kernel_size=3, padding="same",
                        activation="gelu", name="conv1")(seq)
    seq = layers.BatchNormalization(name="conv1_bn")(seq)
    seq = layers.Conv1D(filters=64, kernel_size=3, padding="same",
                        activation="gelu", name="conv2")(seq)
    seq = layers.BatchNormalization(name="conv2_bn")(seq)
    seq = layers.MaxPooling1D(pool_size=2, name="conv_maxpool")(seq)  # (8, 64)

    # Bi-LSTM stack
    seq = layers.Bidirectional(
        layers.LSTM(lstm_units, return_sequences=True),
        name="bilstm_1"
    )(seq)
    seq = layers.Bidirectional(
        layers.LSTM(lstm_units // 2, return_sequences=True),
        name="bilstm_2"
    )(seq)

    # Multi-head self-attention + global pooling
    attn = layers.MultiHeadAttention(
        num_heads=4,
        key_dim=32,
        name="mh_attention"
    )(seq, seq)
    attn = layers.Add(name="attn_residual")([seq, attn])
    attn = layers.LayerNormalization(name="attn_norm")(attn)
    seq_out = layers.GlobalAveragePooling1D(name="seq_global_avg")(attn)

    # ---------------- Parallel MLP tower ----------------
    mlp = layers.Dense(encoder_dim, activation="gelu", name="mlp_dense_1")(enc_out)
    mlp = layers.Dropout(dropout_rate, name="mlp_drop_1")(mlp)
    mlp = layers.Dense(encoder_dim // 2, activation="gelu", name="mlp_dense_2")(mlp)
    mlp = layers.Dropout(dropout_rate, name="mlp_drop_2")(mlp)

    # ---------------- Fusion & Head ----------------
    fused = layers.Concatenate(name="fusion_concat")([seq_out, mlp])
    fused = layers.BatchNormalization(name="fusion_bn")(fused)
    fused = layers.Dropout(dropout_rate, name="fusion_drop_1")(fused)
    fused = layers.Dense(encoder_dim, activation="gelu", name="fusion_dense_1")(fused)
    fused = layers.Dropout(dropout_rate, name="fusion_drop_2")(fused)
    fused = layers.Dense(encoder_dim // 2, activation="gelu", name="fusion_dense_2")(fused)
    fused = layers.Dropout(dropout_rate, name="fusion_drop_3")(fused)

    outputs = layers.Dense(num_classes, activation="softmax", name="output_softmax")(fused)

    model = models.Model(inputs=inputs, outputs=outputs, name="FALCON_ID_v3")

    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

# Training & Evaluation

In [7]:
def train_falcon_v3(dataset_name, X_train, y_train, X_val, y_val, num_classes):
    input_dim = X_train.shape[1]

    print("\n" + "="*30)
    print(f"📌 Dataset: {dataset_name}")
    print("="*30)
    print(f"🚀 Training FALCON-ID v3 for: {dataset_name}")
    print(f"Input dim: {input_dim}, Classes: {num_classes}")

    model = build_falcon_id_model_v3(
        input_dim=input_dim,
        num_classes=num_classes,
        encoder_dim=256,
        encoder_blocks=3,
        lstm_units=128,
        dropout_rate=0.4
    )

    ckpt_path = os.path.join(MODELS_DIR, f"{dataset_name}_best_v3.keras")

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_accuracy",
            patience=3,
            restore_best_weights=True,
            verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_accuracy",
            factor=0.5,
            patience=1,
            mode="max",
            min_lr=1e-5,
            verbose=1
        ),
        tf.keras.callbacks.ModelCheckpoint(
            filepath=ckpt_path,
            monitor="val_accuracy",
            save_best_only=True,
            save_weights_only=False,
            mode="max",
            verbose=1
        )
    ]

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        verbose=1
    )

    full_model_path = os.path.join(
        MODELS_DIR, f"{dataset_name}_FALCON_ID_v3.keras"
    )
    model.save(full_model_path)
    print(f"✔ Model saved → {full_model_path}")

    return model, history

In [8]:
def evaluate_falcon_v3(dataset_name, model, X_test, y_test):
    y_prob = model.predict(X_test, batch_size=BATCH_SIZE)
    y_pred = np.argmax(y_prob, axis=1)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="weighted", zero_division=0)
    rec = recall_score(y_test, y_pred, average="weighted", zero_division=0)
    f1  = f1_score(y_test, y_pred, average="weighted", zero_division=0)

    print(f"\n📊 Evaluation Results — {dataset_name}")
    print(f"Accuracy : {acc:.6f}")
    print(f"Precision: {prec:.6f}")
    print(f"Recall   : {rec:.6f}")
    print(f"F1 Score : {f1:.6f}")

    metrics_path = os.path.join(MODELS_DIR, f"{dataset_name}_metrics_v3.json")
    with open(metrics_path, "w") as f:
        json.dump(
            {
                "dataset": dataset_name,
                "accuracy": float(acc),
                "precision_weighted": float(prec),
                "recall_weighted": float(rec),
                "f1_weighted": float(f1),
                "num_test_samples": int(len(y_test))
            },
            f,
            indent=4
        )
    print(f"✔ Metrics saved → {metrics_path}")

    return acc, prec, rec, f1

# Train & Evaluate on All Four Datasets

In [9]:
results = []

for name, data in all_data.items():
    X_train, y_train = data["X_train"], data["y_train"]
    X_val,   y_val   = data["X_val"],   data["y_val"]
    X_test,  y_test  = data["X_test"],  data["y_test"]

    num_classes = len(np.unique(y_train))

    model, history = train_falcon_v3(
        dataset_name=name,
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        num_classes=num_classes
    )

    acc, prec, rec, f1 = evaluate_falcon_v3(
        dataset_name=name,
        model=model,
        X_test=X_test,
        y_test=y_test
    )

    results.append({
        "dataset": name,
        "accuracy": acc,
        "precision_weighted": prec,
        "recall_weighted": rec,
        "f1_weighted": f1,
        "num_classes": num_classes,
        "num_train_samples": int(X_train.shape[0]),
        "num_val_samples": int(X_val.shape[0]),
        "num_test_samples": int(X_test.shape[0]),
        "latent_dim": int(X_train.shape[1]),
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE
    })


📌 Dataset: CIC_IIoT_2025
🚀 Training FALCON-ID v3 for: CIC_IIoT_2025
Input dim: 64, Classes: 7
Epoch 1/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.3443 - loss: 1.9532
Epoch 1: val_accuracy improved from -inf to 0.64013, saving model to /users/
15/15 ━━━━━━━━━━━━━━━━━━━━ 20s 167ms/step - accuracy: 0.3538 - loss: 1.9208 - val_accuracy: 0.6401 - val_loss: 1.0353 - learning_rate: 0.0010
Epoch 2/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.7354 - loss: 0.6863
Epoch 2: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.

Epoch 2: val_accuracy did not improve from 0.64013
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 47ms/step - accuracy: 0.7363 - loss: 0.6831 - val_accuracy: 0.5791 - val_loss: 1.0712 - learning_rate: 0.0010
Epoch 3/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.8003 - loss: 0.4980
Epoch 3: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.

Epoch 3: val_accuracy did not improve from 0.64013
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 44ms/

# Summary Table

In [10]:
results_df = pd.DataFrame(results)
print("\n==============================")
print("✅ FALCON-ID v3 Local Model Summary")
print("==============================")
display(results_df)


✅ FALCON-ID v3 Local Model Summary


,dataset,accuracy,precision_weighted,recall_weighted,f1_weighted,num_classes,num_train_samples,num_val_samples,num_test_samples,latent_dim,epochs,batch_size
0,CIC_IIoT_2025,0.640343,0.727229,0.640343,0.626508,7,29424,6305,6306,64,10,2048
1,CIC_BCCC_NRC_IoMT_2024,0.973757,0.970459,0.973757,0.968253,15,2369719,507797,507797,64,10,2048
2,CSE_CIC_IDS2018,0.973451,0.965880,0.973451,0.965045,15,6737603,1443772,1443773,64,10,2048
3,FALCON_ID_Combined,0.973206,0.967483,0.973206,0.966023,37,9136746,1957874,1957876,64,10,2048


In [11]:
summary_csv_path = os.path.join(MODELS_DIR, "FALCON_ID_v3_local_results_summary.csv")
results_df.to_csv(summary_csv_path, index=False)
print(f"✔ Summary CSV saved → {summary_csv_path}")

✔ Summary CSV saved → /users/
